## Desafio 3 da Trilha de Visão Computacional — Rastreamento de Objetos

> **Resumo:** Usamos o modo **`track`** da YOLO para não só *detectar* objetos em cada frame, mas
> seguir cada um com um **ID persistente** ao longo do vídeo (object tracking). No fim, salvamos um
> vídeo com as caixas e os IDs desenhados.

> 📝 **Detecção vs Tracking:** a detecção é quadro-a-quadro e "esquece" tudo entre frames. O
> tracking liga as detecções no tempo (com algoritmos tipo ByteTrack/BoT-SORT), mantendo o mesmo
> ID para o mesmo objeto — é o que permite *contar* objetos e analisar trajetórias.

> ⚠️ **GPU recomendada** (Ambiente de execução → GPU T4): processar todos os frames do vídeo é
> bem mais rápido em GPU, embora a YOLO nano também rode em CPU.

Nesse desafio você irá implementar uma solução simples para realizar o rastreamento de objetos.

Para isso você deverá seguir os pasos desse documento atentamente.

### Instalação e Importação da Biblioteca

In [1]:
!pip install ultralytics opencv-python torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.4 MB/s eta 0:00:0000:01


In [2]:
import ultralytics
import cv2
import torch

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Processar dados

In [5]:
# Baixa o modelo
model = ultralytics.YOLO("yolo11n.pt")  # modelo nano da YOLO (treinado no COCO)

# Joga o modelo para o dispositivo de computação
device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU se disponível, senão CPU
model = model.to(device)  # move o modelo para o dispositivo
print(f"Rodando em: {device}")

# processa o vídeo no modo TRACK: detecta + dá um ID persistente a cada objeto entre frames
results = model.track(source="video.mov", conf=0.1, iou=0.7)

Rodando em: cuda

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/239) /content/video.mov: 384x640 1 apple, 1 orange, 49.8ms
video 1/1 (frame 2/239) /content/video.mov: 384x640 1 apple, 1 orange, 18.2ms
video 1/1 (frame 3/239) /content/video.mov: 384x640 1 apple, 1 orange, 12.3ms
video 1/1 (frame 4/239) /content/video.mov: 384x640 1 apple, 1 orange, 11.5ms
video 1/1 (frame 5/239) /content/video.mov: 384x640 1 apple, 1 orange, 11.6ms
video 1/1 (frame 6/239) /content/video.mov: 384x640

### Salvar Vídeo

In [6]:
# Resumo do rastreamento: quantos objetos distintos foram seguidos ao longo do vídeo
all_ids = set()
for r in results:
    if r.boxes is not None and r.boxes.id is not None:
        all_ids.update(r.boxes.id.int().tolist())

print(f"Total de frames processados: {len(results)}")
print(f"Objetos distintos rastreados (IDs únicos): {len(all_ids)}")
print(f"IDs: {sorted(all_ids)}")
print()
print("Amostra dos primeiros frames (nº de objetos rastreados por frame):")
for idx, r in enumerate(results[:10]):
    n = 0 if (r.boxes is None or r.boxes.id is None) else len(r.boxes.id)
    print(f"  Frame {idx:03d}: {n} objetos")

Total de frames processados: 239
Objetos distintos rastreados (IDs únicos): 21
IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]

Amostra dos primeiros frames (nº de objetos rastreados por frame):
  Frame 000: 2 objetos
  Frame 001: 2 objetos
  Frame 002: 2 objetos
  Frame 003: 2 objetos
  Frame 004: 2 objetos
  Frame 005: 4 objetos
  Frame 006: 4 objetos
  Frame 007: 4 objetos
  Frame 008: 4 objetos
  Frame 009: 4 objetos


In [7]:
# Cria o escritor do vídeo de output.
# IMPORTANTE: o tamanho do writer precisa bater com o tamanho dos frames anotados (r.plot()),
# senão o OpenCV salva um vídeo vazio/corrompido. Por isso pegamos o tamanho real do 1º frame.
annotated = results[0].plot()
height, width = annotated.shape[:2]
writer = cv2.VideoWriter("output.avi", cv2.VideoWriter_fourcc(*"XVID"), 20, (width, height))

# Escreve cada frame anotado (caixas + IDs de tracking) no vídeo
for r in results:
    writer.write(r.plot())

writer.release()  # IMPORTANTE: fecha o arquivo — sem isso o vídeo fica incompleto/inválido
print(f"Vídeo salvo: output.avi  ({width}x{height}, {len(results)} frames)")

Vídeo salvo: output.avi  (1920x1080, 239 frames)


### Referência

- https://docs.ultralytics.com/modes/track/